In [1]:
# ----- Load libraries -----

using CSV, JLD2;
using Dates, DataFrames, Statistics;
include("./code/Metropolis-Within-Gibbs/MetropolisWithinGibbs.jl");
using Main.MetropolisWithinGibbs;
using PlotlyJS, ORCA;
pltjs = PlotlyJS;

┌ Warning: Kaledio is not available on this system. Julia will be unable to produce any plots.
└ @ PlotlyBase C:\Users\wrc938\.julia\packages\PlotlyBase\NxSlF\src\kaleido.jl:58
┌ Warning: ORCA.jl has been deprecated and all savefig functionality
│ has been implemented directly in PlotlyBase itself.
│ 
│ By implementing in PlotlyBase.jl, the savefig routines are automatically
│ available to PlotlyJS.jl also.
└ @ ORCA C:\Users\wrc938\.julia\packages\ORCA\U5XaN\src\ORCA.jl:8


In [2]:
titles = ["Real GDP", "Employment", "Unemployment rate", "π", "UoM Expected inflation", "SPF Expected inflation"];
scales = ["log Y*100", "log E*100", "Percent", "Percent", "Percent", "Percent"];

In [3]:
# Load chunk0
res_chunk0 = load("res_two_gap_AR2_6_obs_oos_chunk0.jld2");

# Load information about the out-of-sample period
data          = res_chunk0["data_full"];
date          = res_chunk0["date"];
MNEMONIC      = res_chunk0["MNEMONIC"];
end_presample = res_chunk0["end_presample"];
end_oos       = res_chunk0["end_oos"];
oos_length    = res_chunk0["oos_length"];

In [4]:
# Initialise containers
oos_forecast = Vector{Any}(undef, oos_length)
α_array      = Vector{Any}(undef, oos_length)
σ_array      = Vector{Any}(undef, oos_length)

for i in 1:oos_length
    if i == 1 || i % 10 == 0 || i == oos_length
        println("Reading chunk $(i) (out of $(oos_length))")
    end

    jldopen("res_two_gap_AR2_6_obs_oos_chunk$(i).jld2", "r") do res
        oos_forecast[i] = res["distr_fcst"]  # already an Array
        α_array[i]      = res["distr_α"]
        σ_array[i]      = res["σʸ"]
    end
end

max_h = size(oos_forecast[1], 1)

Reading chunk 1 (out of 79)
Reading chunk 10 (out of 79)
Reading chunk 20 (out of 79)
Reading chunk 30 (out of 79)
Reading chunk 40 (out of 79)
Reading chunk 50 (out of 79)
Reading chunk 60 (out of 79)
Reading chunk 70 (out of 79)
Reading chunk 79 (out of 79)


8

In [5]:
# Initialise point forecasts, random walk forecasts
point_forecast = zeros(oos_length-max_h, size(data,2), max_h) |> Array{Union{Missing, Float64}};
rw_forecast    = zeros(oos_length-max_h, size(data,2), max_h) |> Array{Union{Missing, Float64}};
actual         = zeros(oos_length-max_h, size(data,2), max_h) |> Array{Union{Missing, Float64}};

In [6]:
# Point forecast and random walk
for i=1:oos_length-max_h
    
    # Drift
    d = mean(diff(data[1:end_presample+i-1,:], dims=1), dims=1)[:];

    # Loop over the forecast horizon
    for hz = 1:max_h
        
        # Random walk benchmark
        if hz == 1
            rw_forecast[i, :, hz] = d .+ data[end_presample+i-1, :];
        else
            rw_forecast[i, :, hz] = d .+ rw_forecast[i, :, hz-1]
        end
        
        # Median forecast
        point_forecast[i, :, hz] = median(oos_forecast[i][hz, :, :], dims=2);
        
        # Actual data
        actual[i, :, hz] = data[end_presample+i+hz-1, :];
    end
end

In [7]:
# Compute RMSFE
tc_rmsfe = sqrt.(dropdims(mean((actual - point_forecast).^2, dims=1), dims=1));
rw_rmsfe = sqrt.(dropdims(mean((actual - rw_forecast).^2, dims=1), dims=1));

# RMSFE DataFrame
df_rmsfe = DataFrame(tc_rmsfe./rw_rmsfe,:auto);
rename!(df_rmsfe, Symbol.(["h$(hz)" for hz=1:max_h]))
CSV.write("./csv_output/rmsfe.csv", df_rmsfe);

##### Stability of the common components

In [10]:
figures = Array{Any}(undef, 3);

c1 = "rgba(0, 48, 158, .75)"; 
c2 = "rgba(255, 0, 0, .75)";

titles_sub = ["Efficient Gap", "Cost-Push Cycle", "Common Trend"]
scales_sub = ["", "", ""];

for i=1:3
        
    traces1 = Array{Any}(undef, length(α_array));

    for j=1:length(α_array)
        
        if i==1
            αij = median(α_array[j][1,:,:], dims=2);
        elseif i==2
            αij = median(α_array[j][3,:,:], dims=2);
        elseif i==3
            αij = median(α_array[j][5,:,:], dims=2);
        end
        
        traces1[j] = pltjs.scatter(x=date[1:end-max_h], y=αij[1:end-max_h], name="State", mode="lines", line=attr(width=1), showlegend=false);
    end
    
    traces1 = convert(Array{PlotlyJS.GenericTrace{Dict{Symbol,Any}},1}, traces1)

    layout  = pltjs.Layout(;title=titles_sub[i], titlefont=attr(size=12),
                           xaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, nticks=20, tickangle=-90, zeroline=false),
                           yaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, zeroline=false, titlefont=attr(size=10), title=scales_sub[i]));

    figures[i] = pltjs.plot(traces1, layout);
end

fig = [figures[1]; figures[2]; figures[3]];

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 600;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:3
    fig.plot.layout["annotations"][i][:font][:size] = 12;

end
display(fig)

# savefig(fig, "./img/factor_revisions.pdf", format="pdf");

data: [
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, sho

In [12]:
figures = Array{Any}(undef, 3);

c1 = "rgba(0, 48, 158, .75)"; 
c2 = "rgba(255, 0, 0, .75)";

titles_sub = ["Efficient Gap", "Cost-Push Cycle", "Common Trend"]
scales_sub = ["", "", ""];

for i=1:2
        
    traces1 = Array{Any}(undef, length(α_array));

    for j=1:length(α_array)
        
        if i==1
            αij = median(α_array[j][1,:,:], dims=2);
        elseif i==2
            αij = median(α_array[j][3,:,:], dims=2);
        elseif i==3
            αij = median(α_array[j][5,:,:], dims=2);
        end
        
        traces1[j] = pltjs.scatter(x=date[1:end-max_h], y=αij[1:end-max_h], name="State", mode="lines", line=attr(width=1), showlegend=false);
    end
    
    traces1 = convert(Array{PlotlyJS.GenericTrace{Dict{Symbol,Any}},1}, traces1)

    layout  = pltjs.Layout(;title=titles_sub[i], titlefont=attr(size=12),
                           xaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, nticks=20, tickangle=-90, zeroline=false),
                           yaxis=attr(tickfont_size=10, showgrid=true, linecolor="black", mirror=true, zeroline=false, titlefont=attr(size=10), title=scales_sub[i]));

    figures[i] = pltjs.plot(traces1, layout);
end

fig = [figures[1]; figures[2]];

# Size
fig.plot.layout["width"]  = 800;
fig.plot.layout["height"] = 400;

# Margins
fig.plot.layout["margin"][:b]  = 40;
fig.plot.layout["margin"][:t]  = 40;
fig.plot.layout["margin"][:r]  = 40;
fig.plot.layout["margin"][:l]  = 40;

# Title size
for i=1:2
    fig.plot.layout["annotations"][i][:font][:size] = 12;

end
display(fig)

# savefig(fig, "./img/factor_revisions.pdf", format="pdf");

data: [
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, sho